# Peak calling for specific cases

Using the latest on September 29th, 2025

** ALSO, will include more normal cells (CN > 3) instead of cutting off at 5 to capture 
low concentrating ones **


Run the peak calling algorithm, annotate the peaks, save the number peaks, then plot the peaks. 


1. Load the list of data
2. For each data:
    1. load the data
    2. filter for individual-gene from the summary file
    3. in each file, filter cells to only tumors from the qc files
    4. run the peak caller
    5. save the results
    6. plot the annotated peaks
5. Gather the results in one place


Use the same file that is used to run the main mixture model pipeline.

In [ ]:
from peak_detection_utils import *
os.chdir("/data1/shahs3/users/salehis/ecdna/src/mixture_model")
from mixture_models_pyro_utils import load_data

# Set font to Arial
force_arial()
rcParams['font.family'] = 'Arial'
sc.settings.set_figure_params(dpi_save=300)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Setup the output directories
outDir = '/data1/shahs3/users/salehis/ecdna/results/peak_detection_november_23_2025_regions'

# Setup Original Regions files
oncogene_files = glob('/data1/shahs3/users/leej39/dlp/analysis/normalization/region_cn_final/*.sv-seg-cn.region.cn.csv')

# Load the ecDNA mixture model results
oncogenes = pd.read_csv('/data1/shahs3/users/salehis/ecdna/results/model_embedding_november_07_2025_region/tables/processed_model_results_nov_07_2025.csv')


# Assert that more than one file was found in oncogene_files
assert len(oncogene_files) > 1, f"No oncogene files found in {oncogene_files}"


In [ ]:
# Minimum process
figuresDir, tablesDir, dataDir = setup_dirs(outDir)

oncogenes['individual'] = oncogenes['system_name'].tolist()
oncogenes['gene'] = oncogenes['locus'].tolist()
oncogenes['id'] = oncogenes['individual'].astype(str) + '__' + oncogenes['gene'].astype(str)


In [ ]:
rsync -azvp --relative \
    iris:/data1/shahs3/users/salehis/ecdna/./results/peak_detection_**/figures/*.p* \
    /Users/salehis/Projects/ecdna

In [ ]:
ignore_qc = True
do_plot = False
results = None

minimal_normal = 3  ## New param, include more normal cells with CN > 3


missed = []
MAX_i = 10

for i, item in enumerate(oncogenes.iterrows()):
    item = item[1]
    # if i > MAX_i:
    #     print("Breaking after 10 iterations for testing")
    #     break
    individual = item["individual"]
    # Extract the cn path and qc path from oncogene_files
    try:
        fpath, qc_path = get_paths(
            individual=individual, ignore_qc=ignore_qc, oncogene_files=oncogene_files
        )
        locus = item["gene"]
        if ignore_qc:
            tumor_cells = None
        else:
            tumor_cells = get_tumor_cells(qc_path=qc_path)
        print(f"Processing {fpath} for locus {locus}")
        dataset_name, locus, n_peaks, pickle_path = handle_item(
            fpath=fpath,
            locus=locus,
            verbose=False,
            do_plot=do_plot,
            tumor_cells=tumor_cells,
            figuresDir=figuresDir,
            dataDir=dataDir,
            drop_normals=True,
            minimal_normal=minimal_normal,
        )
        # Create a tmp data.frame to add to the results
        tmp_df = pd.DataFrame(
            {
                "dataset_name": [dataset_name],
                "locus": [locus],
                "n_peaks": [n_peaks],
                "pickle_path": [pickle_path],
            }
        )
        if results is None:
            results = tmp_df
        else:
            results = pd.concat([results, tmp_df], ignore_index=True)
        # print(f"Processed {i+1}/{len(results)}: {dataset_name}, {locus}, n_peaks: {n_peaks}, saved to {pickle_path}")
    except Exception as e:
        print(f"Error processing {individual}, {locus}: {e}")
        missed.append((individual, locus, str(e)))
    

In [ ]:

# sort by number of peaks
results = results.sort_values(by='n_peaks', ascending=False)
# save as tsv in tables
results_save_path = os.path.join(tablesDir, 'peak_detection_results.tsv')
results.to_csv(results_save_path, sep='\t', index=False)
print(f"Saved peak detection results to:\n{results_save_path}")



## How many zeros?

In [ ]:
# Remove ecDNA
ecdna = pd.read_csv('/data1/shahs3/users/salehis/ecdna/results/model_embedding_november_07_2025_region/tables/processed_model_results_nov_07_2025.csv')
# Pick score > .5 
ecdna = ecdna[ecdna['score'] > 0.4]


results_IC = results.copy()
# Remove ecDNA cases from results_IC
results_IC['id'] = results_IC['dataset_name'].astype(str) + '_' + results_IC['locus'].astype(str)
results_IC = results_IC[~results_IC['id'].isin(ecdna['id'].tolist())]

# Compute mean_CN to filter matching Jake's `nrow(east[east$mean_cn > 6,])`
for idx, row in results_IC.iterrows():
    id_ = row['dataset_name']
    individual = row['dataset_name']
    locus = row['locus']
    # load the data
    fpath = get_paths(individual=individual, ignore_qc=True, oncogene_files=oncogene_files)[0]
    vals, _, _ = load_data(
        dat_path=fpath,
        locus=locus,
        do_log=False,
        do_prune_left=False,
        keep_cells=tumor_cells,
        drop_normals=True,
        minimal_normal=minimal_normal,
    )
    # compute the standard deviation
    std_vals = np.std(vals)
    max_cn = np.max(vals)
    # compute mean copy number, ignoring nans
    mean_cn = np.nanmean(vals)
    n_cells = len(vals)
    # Add these as new columns to the zero_peak_comb_new
    # (1) build a parallel data.frame, add the new values to, then concatenate at the vary end
    results_IC.loc[idx, 'std_cn'] = std_vals
    results_IC.loc[idx, 'max_cn'] = max_cn
    results_IC.loc[idx, 'mean_cn'] = mean_cn
    results_IC.loc[idx, 'n_cells'] = n_cells

# save this again
results_IC_save_path = os.path.join(tablesDir, 'peak_detection_results_non_ecdna_samples.tsv')
results_IC.to_csv(results_IC_save_path, sep='\t', index=False)


# Filter down by mean_cn > 6
results_IC_o6 = results_IC[results_IC['mean_cn'] > 6].copy()

results_IC_o6['dataset_name'].value_counts()

# Plot the distribution of n_peaks
results_IC_o6['n_peaks'].value_counts().sort_index()
# PLot
plt.figure(figsize=(4,3))
sns.countplot(data=results_IC_o6, x='n_peaks', order=sorted(results_IC_o6['n_peaks'].unique()))
plt.xlabel('Number of Peaks Detected')
plt.ylabel('Number of Samples')
plt.title('Distribution of Detected Peaks (Non-ecDNA Samples)')
plt.savefig(os.path.join(figuresDir, 'n_peaks_distribution_non_ecdna_samples.png'), dpi=300, bbox_inches='tight')
plt.close()


rsync -azvp --relative \
    iris:/data1/shahs3/users/salehis/ecdna/./results/peak_detection_november_23_2025_regions/figures/*.p* \
    /Users/salehis/Projects/ecdna


# show the zero ones
results_IC_o6_zero = results_IC_o6[results_IC_o6['n_peaks'] == 0]
results_IC_o6_zero['dataset_name'].value_counts()
results_IC_o6_zero['id'].tolist()


## Plot the zero cases 

In [ ]:
ignore_qc = True
do_plot = True
results_zero = None

minimal_normal = 3  ## New param, include more normal cells with CN > 3

tmp_dataDir = os.path.join(outDir, 'data_debug_zero_peaks_min3')
tmp_figuresDir = os.path.join(outDir, 'figures_debug_zero_peaks_min3')


missed = []
MAX_i = 10

for i, item in enumerate(oncogenes.iterrows()):
    item = item[1]
    # if i > MAX_i:
    #     print("Breaking after 10 iterations for testing")
    #     break
    individual = item["individual"]
    # Extract the cn path and qc path from oncogene_files
    try:
        fpath, qc_path = get_paths(
            individual=individual, ignore_qc=ignore_qc, oncogene_files=oncogene_files
        )
        locus = item["gene"]
        id_ = individual + '_' + locus
        if id_ not in results_IC_o6_zero['id'].unique().tolist():
            continue
        if ignore_qc:
            tumor_cells = None
        else:
            tumor_cells = get_tumor_cells(qc_path=qc_path)
        print(f"Processing {fpath} for locus {locus}")
        dataset_name, locus, n_peaks, pickle_path = handle_item(
            fpath=fpath,
            locus=locus,
            verbose=False,
            do_plot=do_plot,
            tumor_cells=tumor_cells,
            figuresDir=tmp_figuresDir,
            dataDir=tmp_dataDir,
            drop_normals=True,
            minimal_normal=minimal_normal,
        )
        # Create a tmp data.frame to add to the results
        tmp_df = pd.DataFrame(
            {
                "dataset_name": [dataset_name],
                "locus": [locus],
                "n_peaks": [n_peaks],
                "pickle_path": [pickle_path],
            }
        )
        if results is None:
            results = tmp_df
        else:
            results_zero = pd.concat([results_zero, tmp_df], ignore_index=True)
        # print(f"Processed {i+1}/{len(results_zero)}: {dataset_name}, {locus}, n_peaks: {n_peaks}, saved to {pickle_path}")
    except Exception as e:
        print(f"Error processing {individual}, {locus}: {e}")
        missed.append((individual, locus, str(e)))
    

In [ ]:
results = pd.read_csv(os.path.join(tablesDir, 'peak_detection_results.tsv'), sep='\t')

In [ ]:
# Load the pickle files, and add a row for each dataset, locus, peak, left_base_position, right_base_position, peak_height
def load_peak_results(pickle_path):
    with open(pickle_path, 'rb') as f:
        results_dict = pickle.load(f)
    return results_dict

detailed_results = None
adding = 0
for i, items in enumerate(results.iterrows()):
    item = items[1]
    pickle_path = item['pickle_path']
    results_dict = load_peak_results(pickle_path)
    if results_dict is None or len(results_dict['left_base_positions']) == 0:
        print(f"Skipping {pickle_path} as results_dict is None")
        continue
    # Add the results to the results data.frame
    peak_id = 0
    for j in range(len(results_dict['left_base_positions'])):
        left_base_position = results_dict['left_base_positions'][j]
        right_base_position = results_dict['right_base_positions'][j]
        peak_height = results_dict['peak_heights_list'][j]
        new_row = {
            'dataset_name': results_dict['dataset_name'],
            'locus': results_dict['locus'],
            'n_peaks': len(results_dict['left_base_positions']),
            'left_base_position': left_base_position,
            'right_base_position': right_base_position,
            'peak_height': peak_height,
            'peak_width': right_base_position - left_base_position,
            'peak_id': peak_id,
            'n_cells': results_dict['n_cells'],
        }
        peak_id += 1
        if detailed_results is None:
            detailed_results = pd.DataFrame([new_row])
        else:
            detailed_results = pd.concat([detailed_results, pd.DataFrame([new_row])], ignore_index=True)
            adding += 1

# Save
detailed_results.to_csv(os.path.join(tablesDir, 'peak_detection_detailed_results.tsv'), sep='\t', index=False)

# sort ascending by peak_height
detailed_results = detailed_results.sort_values(by='peak_height', ascending=True)
detailed_results

In [ ]:
detailed_results = pd.read_csv(os.path.join(tablesDir, 'peak_detection_detailed_results.tsv'), sep='\t')

In [ ]:
detailed_results.sort_values(by='peak_width', ascending=False)

In [ ]:
detailed_results['peak_height_fraction'] = detailed_results['peak_height'] / detailed_results['n_cells']
# Order by peak_height_fraction
detailed_results = detailed_results.sort_values(by='peak_height_fraction', ascending=True)

## Check the 0 peaks cases

In [ ]:
detailed_results = pd.read_csv(os.path.join(tablesDir, 'peak_detection_detailed_results.tsv'), sep='\t')
detailed_results[detailed_results['n_peaks'] == 0]
# This was empty

In [ ]:
# Load the pickle files, and add a row for each dataset, locus, peak, left_base_position, right_base_position, peak_height
def load_peak_results(pickle_path):
    with open(pickle_path, 'rb') as f:
        results_dict = pickle.load(f)
    return results_dict


# Load the results again 
results = pd.read_csv(os.path.join(tablesDir, 'peak_detection_results.tsv'), sep='\t')

# Find datasets where there is no peaks at any locus at all
# for each dataset, just some over n_peaks

zero_peak_datasets = results[['dataset_name', 'n_peaks']].groupby('dataset_name').sum()
# filter down to only those with zero peaks
zero_peak_datasets = zero_peak_datasets[zero_peak_datasets['n_peaks'] == 0]
zero_peak_datasets = zero_peak_datasets.reset_index()



In [ ]:


detailed_results = None
adding = 0
for i, items in enumerate(results.iterrows()):
    item = items[1]
    pickle_path = item['pickle_path']
    results_dict = load_peak_results(pickle_path)
    if results_dict is None or len(results_dict['left_base_positions']) == 0:
        print(f"Skipping {pickle_path} as results_dict is None")
        continue
    # Add the results to the results data.frame
    peak_id = 0
    for j in range(len(results_dict['left_base_positions'])):
        left_base_position = results_dict['left_base_positions'][j]
        right_base_position = results_dict['right_base_positions'][j]
        peak_height = results_dict['peak_heights_list'][j]
        new_row = {
            'dataset_name': results_dict['dataset_name'],
            'locus': results_dict['locus'],
            'n_peaks': len(results_dict['left_base_positions']),
            'left_base_position': left_base_position,
            'right_base_position': right_base_position,
            'peak_height': peak_height,
            'peak_width': right_base_position - left_base_position,
            'peak_id': peak_id,
            'n_cells': results_dict['n_cells'],
        }
        peak_id += 1
        if detailed_results is None:
            detailed_results = pd.DataFrame([new_row])
        else:
            detailed_results = pd.concat([detailed_results, pd.DataFrame([new_row])], ignore_index=True)
            adding += 1

# Save
detailed_results.to_csv(os.path.join(tablesDir, 'peak_detection_detailed_results.tsv'), sep='\t', index=False)

# sort ascending by peak_height
detailed_results = detailed_results.sort_values(by='peak_height', ascending=True)
detailed_results